In [1]:
# Cell 1 — Setup
import sqlite3, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

conn = sqlite3.connect(Path.cwd().parent / "local_store" / "fraud.db")

df = pd.read_sql("""
    SELECT t.id, t.user_id, t.merchant_id, t.device_id,
           t.amount, t.channel, t.ip_address, t.created_at,
           fl.is_fraud
    FROM transactions t
    JOIN fraud_labels fl ON fl.transaction_id = t.id
""", conn)

df["hour"]       = df["created_at"].str[11:13].astype(int)
df["is_fraud"]   = df["is_fraud"].astype(int)
fraud = df[df.is_fraud == 1]
legit = df[df.is_fraud == 0]

print(f"Total:  {len(df):,}")
print(f"Fraud:  {len(fraud):,}  ({len(fraud)/len(df)*100:.1f}%)")
print(f"Legit:  {len(legit):,}  ({len(legit)/len(df)*100:.1f}%)")

DatabaseError: Execution failed on sql '
    SELECT t.id, t.user_id, t.merchant_id, t.device_id,
           t.amount, t.channel, t.ip_address, t.created_at,
           fl.is_fraud
    FROM transactions t
    JOIN fraud_labels fl ON fl.transaction_id = t.id
': no such table: transactions

In [ ]:
# Cell 2 — Class imbalance bar
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["Legitimate", "Fraud"], [len(legit), len(fraud)],
       color=["#1D9E75", "#D85A30"])
ax.set_title("Class distribution")
ax.set_ylabel("Transaction count")
for i, v in enumerate([len(legit), len(fraud)]):
    ax.text(i, v + 50, f"{v:,}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()
print(f"\nImbalance ratio: {len(legit)/len(fraud):.1f}:1")
print("This is why we use scale_pos_weight in XGBoost")

In [ ]:
# Cell 3 — Amount distribution: fraud vs legit
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(legit["amount"], bins=60, color="#1D9E75", alpha=0.7, label="Legit")
axes[0].hist(fraud["amount"], bins=60, color="#D85A30", alpha=0.7, label="Fraud")
axes[0].set_xlabel("Amount (INR)")
axes[0].set_title("Amount distribution (raw)")
axes[0].legend()

axes[1].hist(np.log1p(legit["amount"]), bins=60, color="#1D9E75", alpha=0.7, label="Legit")
axes[1].hist(np.log1p(fraud["amount"]), bins=60, color="#D85A30", alpha=0.7, label="Fraud")
axes[1].set_xlabel("log(Amount)")
axes[1].set_title("Amount distribution (log scale)")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nFraud  — mean: ₹{fraud.amount.mean():,.0f}  median: ₹{fraud.amount.median():,.0f}  max: ₹{fraud.amount.max():,.0f}")
print(f"Legit  — mean: ₹{legit.amount.mean():,.0f}  median: ₹{legit.amount.median():,.0f}  max: ₹{legit.amount.max():,.0f}")
print(f"Ratio:  {fraud.amount.mean()/legit.amount.mean():.1f}×")

In [ ]:
# Cell 4 — Hour of day: when does fraud happen?
fig, ax = plt.subplots(figsize=(12, 4))
hours = range(24)
fraud_by_hour = fraud.groupby("hour").size()
legit_by_hour = legit.groupby("hour").size()

fraud_pct = fraud_by_hour / fraud_by_hour.sum() * 100
legit_pct = legit_by_hour / legit_by_hour.sum() * 100

ax.bar([h - 0.2 for h in hours], [legit_pct.get(h, 0) for h in hours],
       width=0.4, color="#1D9E75", alpha=0.8, label="Legit")
ax.bar([h + 0.2 for h in hours], [fraud_pct.get(h, 0) for h in hours],
       width=0.4, color="#D85A30", alpha=0.8, label="Fraud")
ax.axvspan(-0.5, 4.5, alpha=0.1, color="red", label="Late night zone (0-4am)")
ax.set_xlabel("Hour of day")
ax.set_ylabel("% of transactions")
ax.set_title("Transaction timing: fraud vs legitimate")
ax.set_xticks(hours)
ax.legend()
plt.tight_layout()
plt.show()

late_night_fraud = (fraud.hour < 5).mean() * 100
late_night_legit = (legit.hour < 5).mean() * 100
print(f"\nLate-night fraud rate: {late_night_fraud:.1f}%")
print(f"Late-night legit rate: {late_night_legit:.1f}%")
print(f"Fraud is {late_night_fraud/max(late_night_legit,0.1):.0f}× more likely at night")

In [ ]:
# Cell 5 — IP address signal
FRAUD_IPS = {"185.220.101.5","185.220.101.6","192.42.116.16",
             "199.87.154.255","23.129.64.131"}

fraud_ip_rate = fraud["ip_address"].isin(FRAUD_IPS).mean() * 100
legit_ip_rate = legit["ip_address"].isin(FRAUD_IPS).mean() * 100

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["Legit", "Fraud"], [legit_ip_rate, fraud_ip_rate],
       color=["#1D9E75", "#D85A30"])
ax.set_ylabel("% using Tor/fraud IP")
ax.set_title("Fraud IP usage rate")
for i, v in enumerate([legit_ip_rate, fraud_ip_rate]):
    ax.text(i, v + 0.5, f"{v:.1f}%", ha="center")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 5 — IP address signal
FRAUD_IPS = {"185.220.101.5","185.220.101.6","192.42.116.16",
             "199.87.154.255","23.129.64.131"}

fraud_ip_rate = fraud["ip_address"].isin(FRAUD_IPS).mean() * 100
legit_ip_rate = legit["ip_address"].isin(FRAUD_IPS).mean() * 100

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["Legit", "Fraud"], [legit_ip_rate, fraud_ip_rate],
       color=["#1D9E75", "#D85A30"])
ax.set_ylabel("% using Tor/fraud IP")
ax.set_title("Fraud IP usage rate")
for i, v in enumerate([legit_ip_rate, fraud_ip_rate]):
    ax.text(i, v + 0.5, f"{v:.1f}%", ha="center")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6 — Device signal
BAD_MERCHANTS = {"m_crypto","m_giftcard","m_jewelry","m_luxury"}

fraud_new_dev = fraud["device_id"].str.contains("fraud", na=False).mean() * 100
fraud_bad_mch = fraud["merchant_id"].isin(BAD_MERCHANTS).mean() * 100
legit_bad_mch = legit["merchant_id"].isin(BAD_MERCHANTS).mean() * 100

print("=== Signal strength summary ===\n")
signals = {
    "is_late_night (0-4am)":      ((fraud.hour<5).mean()*100, (legit.hour<5).mean()*100),
    "ip_fraud_history (Tor IP)":  (fraud_ip_rate, legit_ip_rate),
    "is_new_device (d_fraud_*)":  (fraud_new_dev, 0.0),
    "merchant_risk (crypto etc)": (fraud_bad_mch, legit_bad_mch),
}
print(f"{'Signal':<35} {'Fraud %':>8}  {'Legit %':>8}  {'Ratio':>6}")
print("-" * 62)
for name, (f_pct, l_pct) in signals.items():
    ratio = f_pct / max(l_pct, 0.1)
    print(f"  {name:<33} {f_pct:>7.1f}%  {l_pct:>7.1f}%  {ratio:>5.0f}×")